# LeetCode 3945 — Digit Frequency Score bằng Python

Link đề: https://leetcode.com/problems/digit-frequency-score/ · Độ khó: **Easy** · Chủ đề: Math, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def digitFrequencyScore(self, n: int) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: điểm của `n` là tổng `d * freq(d)` trên **các chữ số khác nhau** `d`, với `freq(d)` là số lần `d` xuất hiện trong `n`.
2. Phát hiện **bước đột phá toán học**: `sum(d * freq(d))` chính là **tổng các chữ số** của `n` (mỗi lần xuất hiện của `d` góp đúng `d`).
3. Biết tách chữ số bằng `divmod(n, 10)` / `str(n)` và đếm bằng `Counter` hoặc mảng đếm 10 ô.
4. Viết 4 cách (Counter, mảng đếm, tách chữ số số học, tổng chữ số) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: nhầm với số lượng chữ số khác nhau, cộng `freq` thay vì `d * freq`, quên `n` có thể chứa chữ số `0`.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. **Điểm tần suất chữ số** (digit frequency score) của `n` là tổng, lấy trên mọi chữ số `d` **xuất hiện trong `n`**, của `d * freq(d)`, trong đó `freq(d)` là số lần chữ số `d` xuất hiện. Trả về điểm đó.

| Ví dụ | `n` | Output | Giải thích |
|---|---|---|---|
| 1 | `122` | `5` | Chữ số `1` xuất hiện 1 lần -> `1*1 = 1`; chữ số `2` xuất hiện 2 lần -> `2*2 = 4`; tổng `5` |
| 2 | `101` | `2` | Chữ số `0` xuất hiện 1 lần -> `0*1 = 0`; chữ số `1` xuất hiện 2 lần -> `1*2 = 2`; tổng `2` |

Constraints: `1 <= n <= 10^9`.

Chú ý: tổng chỉ lấy theo **các chữ số khác nhau** `d`, mỗi `d` một lần, nhân với số lần nó xuất hiện.

In [1]:
from collections import Counter

for n in (122, 101):
    freq = Counter(str(n))
    parts = [f"{d}*{c}" for d, c in freq.items()]
    total = sum(int(d) * c for d, c in freq.items())
    print(f"n={n}: freq={dict(freq)}, điểm = " + " + ".join(parts) + f" = {total}")

n=122: freq={'1': 1, '2': 2}, điểm = 1*1 + 2*2 = 5
n=101: freq={'1': 2, '0': 1}, điểm = 1*2 + 0*1 = 2


## 3. Kiến thức nền

### 3.1. Tách chữ số của số nguyên bằng `divmod`

`divmod(n, 10)` trả về `(n // 10, n % 10)`: phần thương (bỏ chữ số cuối) và chữ số cuối cùng. Lặp tới khi `n == 0` là cách tách mọi chữ số (từ phải sang trái) mà không dùng chuỗi.

In [2]:
n = 122
digits = []
while n:
    n, d = divmod(n, 10)
    digits.append(d)
print(digits)                 # [2, 2, 1]: từ chữ số cuối lên

[2, 2, 1]


### 3.2. Đổi số sang chuỗi: `str(n)`

`str(122) = "122"`; duyệt từng ký tự rồi `int(ch)` thành số. Ngắn hơn, nhưng tạo chuỗi trung gian. Với `n <= 10^9` chỉ tối đa 10 chữ số nên không đáng kể.

In [3]:
n = 4051
print(str(n), [int(ch) for ch in str(n)], len(str(n)))

4051 [4, 0, 5, 1] 4


### 3.3. `collections.Counter`

`Counter(iterable)` đếm số lần xuất hiện của từng phần tử, trả về một từ điển `phần_tử -> số lần`. `Counter("122")` cho `{'1': 1, '2': 2}`. Truy cập khóa không có trả về `0` (không lỗi).

In [4]:
from collections import Counter
c = Counter("122")
print(c, c["1"], c["2"], c["9"])

Counter({'2': 2, '1': 1}) 1 2 0


### 3.4. Mảng đếm kích thước 10

Chữ số chỉ có 10 giá trị `0..9`, nên thay từ điển bằng list `cnt = [0] * 10` và tăng `cnt[d] += 1`. Truy cập `O(1)`, không cần import.

In [5]:
cnt = [0] * 10
for ch in "101":
    cnt[int(ch)] += 1
print(cnt)

[1, 2, 0, 0, 0, 0, 0, 0, 0, 0]


### 3.5. `sum(...)` với biểu thức sinh (generator expression)

`sum(d * c for d, c in cnt.items())` cộng dồn mà không tạo list trung gian. Với `enumerate(cnt)` ta có cả chỉ số `d` lẫn đếm `c`.

In [6]:
cnt = [0, 2, 1, 0, 0, 0, 0, 0, 0, 0]
print(sum(d * c for d, c in enumerate(cnt)))

4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm đúng theo định nghĩa: đếm tần suất từng chữ số, rồi với mỗi chữ số khác nhau `d` cộng `d * freq(d)`.

Tính tay `n = 122`: chữ số `1` (1 lần), `2` (2 lần) -> `1*1 + 2*2 = 5`.
Tính tay `n = 4051`: `4,0,5,1` mỗi chữ số 1 lần -> `4 + 0 + 5 + 1 = 10`.
Tính tay `n = 3330`: `3` xuất hiện 3 lần, `0` một lần -> `3*3 + 0*1 = 9`.

| `n` | tần suất | điểm |
|---|---|---|
| `122` | `1:1, 2:2` | `1 + 4 = 5` |
| `101` | `0:1, 1:2` | `0 + 2 = 2` |
| `4051` | `4:1, 0:1, 5:1, 1:1` | `10` |
| `3330` | `3:3, 0:1` | `9` |

Nhìn bảng: điểm bằng ... tổng các chữ số của `n`? `1+2+2 = 5`, `1+0+1 = 2`, `4+0+5+1 = 10`, `3+3+3+0 = 9`. Đúng cả bốn! Ta sẽ giải thích vì sao ở mục 6.

In [7]:
def score_def(n):
    freq = Counter(str(n))
    return sum(int(d) * c for d, c in freq.items())

def digit_sum(n):
    return sum(int(ch) for ch in str(n))

for n in (122, 101, 4051, 3330):
    print(n, score_def(n), digit_sum(n))

122 5 5
101 2 2
4051 10 10
3330 9 9


## 5. Cách 1 — Brute Force / baseline: `Counter` đúng theo định nghĩa

1. Đổi `n` sang chuỗi, đếm bằng `Counter`.
2. Với mỗi cặp `(chữ số d, số lần c)`, cộng `int(d) * c`.

Đây là bản dịch trực tiếp của đề, dễ chứng minh đúng. "Nút thắt" duy nhất là nó **dựng cả bảng tần suất** trong khi có thể đơn giản hơn nhiều.

In [8]:
from collections import Counter

class SolutionCounter:
    def digitFrequencyScore(self, n):
        freq = Counter(str(n))
        total = 0
        for d, c in freq.items():
            total += int(d) * c
        return total

print(SolutionCounter().digitFrequencyScore(122), SolutionCounter().digitFrequencyScore(101))

5 2


**Độ phức tạp:** Time `O(L)` với `L = số chữ số của n` (`L <= 10`), Space `O(1)` (tối đa 10 khóa).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** nhóm các lần xuất hiện theo chữ số rồi nhân thực chất chỉ là **cộng dồn từng lần xuất hiện**:

`d * freq(d) = d + d + ... + d` (`freq(d)` số hạng) = tổng của `d` **qua từng lần xuất hiện** của `d` trong `n`.

Cộng trên mọi `d` khác nhau, ta cộng **từng chữ số của `n` đúng một lần**: tức là **tổng các chữ số của `n`**.

`score(n) = sum(d * freq(d)) = d_1 + d_2 + ... + d_L` (với `d_i` là các chữ số của `n`).

**Quy tắc:** không cần đếm tần suất; chỉ cần tính tổng chữ số, ví dụ `n = 122 -> 1 + 2 + 2 = 5`.

**Invariant:** vòng lặp `while n: n, d = divmod(n, 10); total += d` giữ `total = tổng các chữ số đã bóc`, và `n` là phần chưa xử lý.

## 7. Các cách tiếp theo

### Cách 2 — Mảng đếm 10 ô

Không dùng chuỗi: bóc chữ số bằng `divmod`, tăng `cnt[d]`, rồi cộng `d * cnt[d]`. Vẫn theo định nghĩa nhưng nhanh và không import.

In [9]:
class SolutionArray:
    def digitFrequencyScore(self, n):
        cnt = [0] * 10
        while n:
            n, d = divmod(n, 10)
            cnt[d] += 1
        return sum(d * c for d, c in enumerate(cnt))

print(SolutionArray().digitFrequencyScore(122), SolutionArray().digitFrequencyScore(101))

5 2


**Độ phức tạp:** Time `O(L)`, Space `O(1)` (10 ô).

### Cách 3 — Tách chữ số bằng chuỗi rồi cộng trực tiếp

Dựa vào bước đột phá: `sum(map(int, str(n)))`.

In [10]:
class SolutionStringSum:
    def digitFrequencyScore(self, n):
        return sum(map(int, str(n)))

print(SolutionStringSum().digitFrequencyScore(122), SolutionStringSum().digitFrequencyScore(101))

5 2


**Độ phức tạp:** Time `O(L)`, Space `O(L)` (chuỗi `str(n)`).

### Cách 4 — Tổng chữ số bằng số học thuần

Không chuỗi, không mảng: `total += n % 10; n //= 10`. Đây là bản tiết kiệm nhất.

In [11]:
class SolutionMath:
    def digitFrequencyScore(self, n):
        total = 0
        while n > 0:
            total += n % 10
            n //= 10
        return total

print(SolutionMath().digitFrequencyScore(122), SolutionMath().digitFrequencyScore(101))

5 2


**Độ phức tạp:** Time `O(L)`, Space `O(1)`.

Mẹo kiểm tra nhanh: tổng chữ số đồng dư với `n` theo modulo 9 (`n % 9 == digit_sum % 9`), một **kiểm tra chéo** hữu ích để phát hiện lỗi.

### Cách 5 — Đệ quy theo chữ số cuối

`score(n) = n % 10 + score(n // 10)` với `score(0) = 0`. Độ sâu đệ quy chỉ bằng số chữ số (`<= 10`) nên an toàn. Cách này giúp thấy rõ **tính chất cộng dồn**: điểm của `n` bằng chữ số cuối cộng điểm của phần còn lại.

In [12]:
class SolutionRecursive:
    def digitFrequencyScore(self, n):
        if n == 0:
            return 0
        return n % 10 + self.digitFrequencyScore(n // 10)

print(SolutionRecursive().digitFrequencyScore(122), SolutionRecursive().digitFrequencyScore(101))

5 2


**Độ phức tạp:** Time `O(L)`, Space `O(L)` cho ngăn xếp đệ quy (`L <= 10`).

## 8. TRACE từng bước

Trace Cách 4 trên `n = 4051`:

| lần lặp | `n` trước | `n % 10` | `total` sau | `n` sau (`n // 10`) |
|---|---|---|---|---|
| 1 | 4051 | 1 | 1 | 405 |
| 2 | 405 | 5 | 6 | 40 |
| 3 | 40 | 0 | 6 | 4 |
| 4 | 4 | 4 | 10 | 0 |

In [13]:
def trace_math(n):
    print("n =", n)
    total, step = 0, 0
    while n > 0:
        step += 1
        d = n % 10
        total += d
        print(f"  lần {step}: n={n}, chữ số cuối={d}, total={total}, n//10={n // 10}")
        n //= 10
    return total

assert trace_math(4051) == 10
assert trace_math(122) == 5

n = 4051
  lần 1: n=4051, chữ số cuối=1, total=1, n//10=405
  lần 2: n=405, chữ số cuối=5, total=6, n//10=40
  lần 3: n=40, chữ số cuối=0, total=6, n//10=4
  lần 4: n=4, chữ số cuối=4, total=10, n//10=0
n = 122
  lần 1: n=122, chữ số cuối=2, total=2, n//10=12
  lần 2: n=12, chữ số cuối=2, total=4, n//10=1
  lần 3: n=1, chữ số cuối=1, total=5, n//10=0


In [14]:
def trace_counter(n):
    cnt = [0] * 10
    m = n
    while m:
        m, d = divmod(m, 10)
        cnt[d] += 1
    print("cnt =", cnt)
    total = 0
    for d, c in enumerate(cnt):
        if c:
            print(f"  chữ số {d}: xuất hiện {c} lần -> góp {d}*{c} = {d * c}")
            total += d * c
    print("điểm =", total)
    return total

assert trace_counter(3330) == 9
assert trace_counter(101) == 2

cnt = [1, 0, 0, 3, 0, 0, 0, 0, 0, 0]
  chữ số 0: xuất hiện 1 lần -> góp 0*1 = 0
  chữ số 3: xuất hiện 3 lần -> góp 3*3 = 9
điểm = 9
cnt = [1, 2, 0, 0, 0, 0, 0, 0, 0, 0]
  chữ số 0: xuất hiện 1 lần -> góp 0*1 = 0
  chữ số 1: xuất hiện 2 lần -> góp 1*2 = 2
điểm = 2


## 9. Code nộp LeetCode

Dùng Cách 4: tổng chữ số bằng số học.

In [15]:
from typing import List, Optional


class Solution:
    def digitFrequencyScore(self, n: int) -> int:
        total = 0
        while n > 0:
            total += n % 10
            n //= 10
        return total


print(Solution().digitFrequencyScore(122), Solution().digitFrequencyScore(101))

5 2


## 10. Vì sao thuật toán đúng

Gọi các chữ số của `n` là `d_1, d_2, ..., d_L` (có thể trùng nhau) và `D` là tập các chữ số **khác nhau**.

1. Với mỗi `d` trong `D`, `freq(d)` là số chỉ số `i` mà `d_i = d`. Nên `d * freq(d) = sum_{i : d_i = d} d_i`.
2. Mỗi chỉ số `i` thuộc **đúng một** nhóm `{i : d_i = d}` (nhóm của chính `d_i`). Vậy cộng `d * freq(d)` trên mọi `d` thuộc `D` bằng cộng `d_i` trên mọi `i`, tức `d_1 + ... + d_L`.
3. Vòng lặp `while n > 0` lấy lần lượt chữ số cuối `n % 10` rồi bỏ nó bằng `n //= 10`; dừng khi hết chữ số. Nó cộng đúng mỗi chữ số một lần.
4. Chữ số `0` góp `0 * freq(0) = 0` nên bỏ qua hay cộng đều không ảnh hưởng.

Vì vậy Cách 1, 2 (theo định nghĩa) và Cách 3, 4 (tổng chữ số) cho **cùng một số**.

## 11. Phân tích Time / Space Complexity

Gọi `L` là số chữ số của `n`; `L = floor(log10 n) + 1 <= 10` do `n <= 10^9`.

- Cách 1 (`Counter`): Time `O(L)`, Space `O(1)`.
- Cách 2 (mảng 10 ô): Time `O(L + 10)`, Space `O(1)`.
- Cách 3 (`str` + `sum`): Time `O(L)`, Space `O(L)`.
- Cách 4 (nộp): Time `O(L) = O(log n)`, Space `O(1)`.

Không thể nhanh hơn `O(L)` vì mỗi chữ số phải được đọc. Với `n <= 10^9` toàn bộ chỉ vài chục phép tính.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [16]:
import random

solutions = [SolutionCounter(), SolutionArray(), SolutionStringSum(), SolutionMath(), SolutionRecursive(), Solution()]

fixed = [
    (122, 5), (101, 2), (1, 1), (9, 9), (10, 1), (100, 1), (4051, 10), (3330, 9),
    (999999999, 81), (10 ** 9, 1), (123456789, 45), (111111111, 9),
    (505, 10), (900, 9),
]
for n, expected in fixed:
    for sol in solutions:
        assert sol.digitFrequencyScore(n) == expected, (type(sol).__name__, n)
print("test cố định OK:", len(fixed), "ca")

random.seed(3945)
for _ in range(1000):
    n = random.choice([random.randint(1, 100), random.randint(1, 10 ** 9), random.randint(10 ** 8, 10 ** 9)])
    expected = score_def(n)                       # trọng tài theo định nghĩa (Counter)
    for sol in solutions:
        assert sol.digitFrequencyScore(n) == expected, (type(sol).__name__, n)
print("test ngẫu nhiên OK")

test cố định OK: 14 ca
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Điểm đồng dư với `n` theo modulo 9 (`score % 9 == n % 9`). (2) Hoán vị chữ số (không dẫn đầu bằng `0`) không đổi điểm. (3) Điểm `<= 9 * L` và `>= 1` (vì `n >= 1`). (4) `score(n * 10) == score(n)`: thêm chữ số `0` không đổi điểm.

In [17]:
random.seed(39450)
for _ in range(500):
    n = random.randint(1, 10 ** 9)
    s = Solution().digitFrequencyScore(n)
    assert s % 9 == n % 9
    L = len(str(n))
    assert 1 <= s <= 9 * L
    digits = list(str(n)); random.shuffle(digits)
    if digits[0] != "0":
        assert Solution().digitFrequencyScore(int("".join(digits))) == s
    if n * 10 <= 10 ** 9:
        assert Solution().digitFrequencyScore(n * 10) == s
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cộng số **lần xuất hiện** thay vì `d * freq(d)`

Tổng `freq(d)` chỉ là số chữ số của `n`.

In [18]:
def wrong_sum_freq(n):
    return sum(Counter(str(n)).values())                     # SAI: chỉ đếm số chữ số

print(wrong_sum_freq(122), "!=", Solution().digitFrequencyScore(122))
assert wrong_sum_freq(122) == 3 and Solution().digitFrequencyScore(122) == 5

3 != 5


### Lỗi 2: chỉ cộng các chữ số **khác nhau** một lần (`sum(set(...))`)

`set` bỏ trùng: `122 -> {1, 2}` -> `3`, nhưng điểm đúng là `5` vì chữ số `2` xuất hiện 2 lần nên góp `2 * 2`.

In [19]:
def wrong_set_sum(n):
    return sum(int(d) for d in set(str(n)))                  # SAI: bỏ qua tần suất

print(wrong_set_sum(122), "!=", Solution().digitFrequencyScore(122))
assert wrong_set_sum(122) == 3 and wrong_set_sum(101) == 1 and Solution().digitFrequencyScore(101) == 2

3 != 5


### Lỗi 3: nhầm với `d + freq(d)`, hoặc `freq(d) ** 2`

Đọc đề kỹ: `d * freq(d)` là tích. Cộng `d + freq(d)` hay bình phương tần suất cho kết quả khác.

In [20]:
def wrong_add(n):
    return sum(int(d) + c for d, c in Counter(str(n)).items())   # SAI: cộng thay vì nhân

print(wrong_add(122), "!=", Solution().digitFrequencyScore(122))
assert wrong_add(122) == (1 + 1) + (2 + 2) == 6 and Solution().digitFrequencyScore(122) == 5

6 != 5


### Lỗi 4: dùng `int` làm khóa nhưng trộn với ký tự (`str`)

Nếu đếm theo ký tự `'1'` rồi nhân luôn với ký tự (không `int`) thì `'1' * 2 = '11'` (lặp chuỗi), cộng chuỗi và số gây lỗi.

In [21]:
def wrong_no_int(n):
    total = 0
    for d, c in Counter(str(n)).items():
        total += d * c                                       # SAI: d là chuỗi, d * c là lặp chuỗi
    return total

try:
    wrong_no_int(122)
except TypeError as e:
    print("TypeError:", e)
assert Solution().digitFrequencyScore(122) == 5

TypeError: unsupported operand type(s) for +=: 'int' and 'str'


### Lỗi 5: vòng lặp `while n:` làm hỏng `n` rồi dùng lại

Sau `while n: n //= 10`, `n` còn `0`. Nếu sau đó cần dùng `n` gốc (ví dụ để đếm) sẽ sai. Hãy dùng bản sao `m = n`.

In [22]:
def wrong_reuse(n):
    total = 0
    while n:
        total += n % 10
        n //= 10
    return total, n                                          # n đã về 0, không còn là n gốc

print(wrong_reuse(122))
assert wrong_reuse(122) == (5, 0) and Solution().digitFrequencyScore(122) == 5

(5, 0)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy tổng dạng `sum(giá_trị * số_lần)` trên các giá trị **khác nhau** -> thường chính là tổng của **mọi phần tử** (gom nhóm rồi nhân = cộng từng phần tử).
- Tách chữ số: `divmod(n, 10)` (số học) hoặc `str(n)` (chuỗi); số học tiết kiệm bộ nhớ, chuỗi ngắn gọn.
- Chữ số chỉ có 10 giá trị -> **mảng đếm `[0] * 10`** thay cho từ điển.
- Kiểm tra chéo: `digit_sum(n) % 9 == n % 9` (đồng dư modulo 9).
- Nếu đề thay bằng "chỉ tính các chữ số xuất hiện nhiều hơn một lần" hay "`d ** freq`", **không** còn là tổng chữ số: phải đếm thật.
- Đọc constraints: `n <= 10^9` có tối đa 10 chữ số -> mọi cách đều tức thì.

## 15. Biến thể của bài

1. **Add Digits** (258), **Sum of Digits of String After Convert** (1945): các bài tổng chữ số.
2. **Happy Number** (202): lặp tổng bình phương các chữ số.
3. **Count Number of Distinct Integers After Reverse Operations** (2442): thao tác chữ số.
4. **Check Good Integer** (3959, cùng đợt): dùng tổng chữ số và tổng bình phương.
5. Biến thể: điểm = `sum(d * freq(d))` chỉ với `d` xuất hiện **hơn một lần** (cần đếm thật).
6. Biến thể: `n` lớn (chuỗi rất dài `10^5` chữ số): vẫn tổng chữ số, nhưng đọc `n` dưới dạng chuỗi.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `Counter` | đếm tần suất rồi `d * c` | `O(L)` | `O(1)` | Bám sát đề, dễ đối chiếu |
| 2. Mảng đếm 10 ô | `cnt[d] += 1` | `O(L)` | `O(1)` | Đếm chữ số, không import |
| 3. `sum(map(int, str(n)))` | tổng chữ số bằng chuỗi | `O(L)` | `O(L)` | Code một dòng |
| 4. `% 10`, `// 10` | tổng chữ số bằng số học | `O(L)` | `O(1)` | **Nộp bài**, tiết kiệm nhất |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().digitFrequencyScore(777)` và `Solution().digitFrequencyScore(2020)` ra gì?

**Bài 2 (tính tay):** với `n = 9081`, liệt kê tần suất từng chữ số, `d * freq(d)` và tổng.

**Bài 3 (điền TODO):** hoàn thành `score_only_repeated(n)`: tổng `d * freq(d)` chỉ lấy các chữ số xuất hiện **từ 2 lần trở lên**.

**Bài 4 (tự viết lại):** viết `digit_frequency_table(n)` trả về list `cnt` độ dài 10 với `cnt[d]` là số lần `d` xuất hiện trong `n`.

**Bài 5 (mini challenge):** viết `has_equal_score(a, b)`: hai số có cùng điểm hay không; rồi tìm số nhỏ nhất `> a` có cùng điểm với `a`.

In [23]:
# Bài 1: đoán rồi chạy
print(Solution().digitFrequencyScore(777), Solution().digitFrequencyScore(2020))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def score_only_repeated_todo(n):
    # TODO: đếm tần suất, chỉ cộng d * c khi c >= 2
    return 0

21 4


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `777` có chữ số `7` xuất hiện 3 lần. `2020`: chữ số `2` xuất hiện 2 lần, `0` xuất hiện 2 lần. Bài 2: bốn chữ số khác nhau, mỗi chữ số một lần.

**Mức 2 (vừa):** Bài 3: dùng `Counter(str(n))`, thêm điều kiện `if c >= 2`. Bài 4: mảng `[0] * 10` và `divmod`. Bài 5: dùng tổng chữ số, vòng lặp `m = a + 1, a + 2, ...`.

**Mức 3 (gần đáp án):** Bài 1: `21` và `4`. Bài 2: `9+0+8+1 = 18`. Bài 3: `122 -> 4` (chỉ chữ số `2` lặp), `123 -> 0`. Bài 4: `101 -> [1, 2, 0, ..., 0]`. Bài 5: `a = 19 -> 28` (tổng chữ số 10).

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().digitFrequencyScore(777) == 21 and Solution().digitFrequencyScore(2020) == 4
assert Solution().digitFrequencyScore(9081) == 18 and score_def(9081) == 18

# Bài 3
def score_only_repeated(n):
    return sum(int(d) * c for d, c in Counter(str(n)).items() if c >= 2)

assert score_only_repeated(122) == 4 and score_only_repeated(123) == 0 and score_only_repeated(777) == 21

# Bài 4
def digit_frequency_table(n):
    cnt = [0] * 10
    while n:
        n, d = divmod(n, 10)
        cnt[d] += 1
    return cnt

assert digit_frequency_table(101) == [1, 2, 0, 0, 0, 0, 0, 0, 0, 0]
assert sum(digit_frequency_table(123456789)) == 9

# Bài 5
def has_equal_score(a, b):
    return Solution().digitFrequencyScore(a) == Solution().digitFrequencyScore(b)

def next_equal_score(a):
    m = a + 1
    while not has_equal_score(a, m):
        m += 1
    return m

assert has_equal_score(19, 28) and not has_equal_score(19, 29)
assert next_equal_score(19) == 28 and next_equal_score(1) == 10
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def digitFrequencyScore(self, n: int) -> int:
        total = 0
        while n > 0:
            total += n % 10         # chữ số cuối
            n //= 10                # bỏ chữ số cuối
        return total                # sum(d * freq(d)) == tổng chữ số
```

Khung chung "tách chữ số": `while n: n, d = divmod(n, 10); ...`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Đã chứng minh/giải thích vì sao `sum(d * freq(d))` bằng tổng chữ số (mỗi lần xuất hiện góp đúng `d`).
- [ ] Đã thử `n = 1`, `n = 10`, `n = 10**9` (chữ số `0` không ảnh hưởng).